In [2]:
!pip install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00


In [3]:
import torch
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
import torch.nn as nn

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Initialize the model, preprocessing function and tokenizer

In [5]:
from clip_zeroshot import build_and_cache_text_features, build_and_cache_image_features, top_k_accuracy, load_cached_features

In [6]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


# Base to New Evalutation

In [7]:
from coop import PromptLearner, TextEncoderWrapper

### Download and prepare the caltech data

In [8]:
mkdir data features

In [9]:
caltech_dataset = datasets.Caltech101(root='./data', download=True, transform=preprocess)

100%|██████████| 137M/137M [00:11<00:00, 11.5MB/s]


In [10]:
caltech_classes = [cls.replace('_', ' ') for cls in caltech_dataset.categories]
print(caltech_classes[:5])
print(len(caltech_classes))

['Faces', 'Faces easy', 'Leopards', 'Motorbikes', 'accordion']
101


In [11]:
# Split the classes in the middle
base_classes = caltech_classes[:51]
new_classes = caltech_classes[51:]

print(len(base_classes))
print(len(new_classes))

51
50


In [12]:
from collections import defaultdict

base_classes_to_indices = defaultdict(list)
new_classes_to_indices = defaultdict(list)

for idx in range(len(caltech_dataset)):
  label = caltech_dataset[idx][1]
  if label < 51:
    base_classes_to_indices[label].append(idx)
  else:
    new_classes_to_indices[label].append(idx)

In [13]:
print(len(base_classes_to_indices))
print(len(new_classes_to_indices))

51
50


In [14]:
import random

few_shot_indices = []

for label, indices in base_classes_to_indices.items():
  sampled = random.sample(indices, 16)
  few_shot_indices.extend(sampled)

In [15]:
few_shot_indices[:5]

[293, 135, 127, 145, 424]

In [16]:
from torch.utils.data import Subset

few_shot_dataset = Subset(caltech_dataset, few_shot_indices)

In [17]:
len(few_shot_dataset)

816

In [18]:
eval_indices = []

for label, indices in new_classes_to_indices.items():
  eval_indices.extend(indices)

eval_dataset = Subset(caltech_dataset, eval_indices)

In [19]:
len(eval_dataset)

3063

In [20]:
raw_few_shot_loader = DataLoader(few_shot_dataset, batch_size=32, shuffle=True)
raw_eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

### Build the text features

In [21]:
base_prompt_learner = PromptLearner(clip_model=model, device=device, n_ctx=4, tokenizer=tokenizer, ctx_dim=512, class_names=base_classes).to(device)
prompt, tok_prompt = base_prompt_learner()

In [22]:
text_encoder = TextEncoderWrapper(model)

### Build the image features

In [23]:
cache_img_train = build_and_cache_image_features(model, device, raw_few_shot_loader, './features', 'caltech_img_train_base')

  0%|          | 0/26 [00:00<?, ?it/s]

Image features and labels has been saved at ./features/caltech_img_train_base.pt


In [24]:
from torch.utils.data import TensorDataset

img_features_dataset = TensorDataset(cache_img_train['image_features'], cache_img_train['labels'])
train_loader = DataLoader(img_features_dataset, batch_size=32, shuffle=True)

### Training Loop

In [25]:
for param in model.parameters():
  param.requires_grad_(False)

In [26]:
from tqdm.notebook import tqdm
import torch.nn.functional as F

epochs = 10
optimizer = torch.optim.Adam(base_prompt_learner.parameters(), lr=0.002)
num_ctx = 4
ctx_dim = 312
logit_scale = model.logit_scale.exp()

for epoch in range(epochs+1):
  total_loss = 0
  for img_feat, labels in tqdm(train_loader):
    img_feat = img_feat.to(device)
    labels = labels.to(device)

    prompts, tok_prompts = base_prompt_learner()
    text_features = text_encoder(prompts, tok_prompts)
    text_features = text_features / text_features.norm(dim=-1,keepdim=True)

    logits = logit_scale * img_feat @ text_features.t()
    loss = F.cross_entropy(logits, labels)
    total_loss += loss.item()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  epoch_loss = total_loss / len(train_loader)
  print(f"epoch : {epoch + 1}, loss : {epoch_loss: .4f}")

  0%|          | 0/26 [00:00<?, ?it/s]

epoch : 1, loss :  0.2177


  0%|          | 0/26 [00:00<?, ?it/s]

epoch : 2, loss :  0.1535


  0%|          | 0/26 [00:00<?, ?it/s]

...

epoch : 9, loss :  0.0846


  0%|          | 0/26 [00:00<?, ?it/s]

epoch : 10, loss :  0.0838


  0%|          | 0/26 [00:00<?, ?it/s]

epoch : 11, loss :  0.0758


### Evaluation

In [27]:
new_prompt_learner = PromptLearner(clip_model=model, device=device, n_ctx=4, tokenizer=tokenizer, ctx_dim=512, class_names=new_classes).to(device)

In [28]:
new_prompt_learner.ctx.data = base_prompt_learner.ctx.data.clone()

In [29]:
eval_cached = build_and_cache_image_features(model, device, raw_eval_loader, './features', "caltech_eval_new")

eval_feature_dataset = TensorDataset(eval_cached["image_features"], eval_cached["labels"])
test_loader = DataLoader(eval_feature_dataset, batch_size=32, shuffle=False)

  0%|          | 0/96 [00:00<?, ?it/s]

Image features and labels has been saved at ./features/caltech_eval_new.pt


In [30]:
new_prompt_learner.eval()

with torch.no_grad():
    prompts, tokenized = new_prompt_learner()
    text_features = text_encoder(prompts, tokenized)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    correct = 0
    total = 0
    for image_features, labels in test_loader:
        image_features = image_features.to(device)
        labels = labels.to(device)
        labels = labels - 51

        logits = image_features @ text_features.t()
        preds = logits.argmax(dim=-1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"Test accuracy: {accuracy:.2f}")

Test accuracy: 93.24


In [31]:
# base eval = base-class images NOT used in few-shot training
base_all_indices = []
for label, indices in base_classes_to_indices.items():
    base_all_indices.extend(indices)
base_eval_indices = list(set(base_all_indices) - set(few_shot_indices))
base_eval_dataset = Subset(caltech_dataset, base_eval_indices)

In [32]:
raw_base_eval_loader = DataLoader(base_eval_dataset, batch_size=32, shuffle=True)

In [33]:
base_eval_cache = build_and_cache_image_features(model, device, raw_base_eval_loader, './features', 'caltech_img_eval_base')

  0%|          | 0/150 [00:00<?, ?it/s]

Image features and labels has been saved at ./features/caltech_img_eval_base.pt


In [34]:
eval_img_features_dataset = TensorDataset(base_eval_cache['image_features'], base_eval_cache['labels'])
eval_base_loader = DataLoader(eval_img_features_dataset, batch_size=32, shuffle=True)

In [35]:
print(base_eval_cache["labels"].min(), eval_cached["labels"].max())  # expect 51, 100

tensor(0) tensor(100)


In [36]:
print(eval_cached["labels"].min(), eval_cached["labels"].max())  # want: 51, 100

tensor(51) tensor(100)


In [37]:
base_prompt_learner.eval()
with torch.no_grad():
    prompts, tokenized = base_prompt_learner()
    text_features = text_encoder(prompts, tokenized)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    correct = 0
    total = 0
    for image_features, labels in eval_base_loader:
        image_features = image_features.to(device)
        labels = labels.to(device)

        logits = image_features @ text_features.t()
        preds = logits.argmax(dim=-1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"Test accuracy: {accuracy:.2f}")

Test accuracy: 90.29
